In [42]:
import torch, numpy as np, json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import hf_hub_download
import onnxruntime as ort

In [43]:
MODEL_REVISION = "v0.4.1-intl"
HG_MODEL = "livekit/turn-detector"
ONNX_FILENAME = "model_q8.onnx"
onnx_path = hf_hub_download(HG_MODEL, ONNX_FILENAME, subfolder="onnx",
                           revision=MODEL_REVISION, local_files_only=True)
tok = AutoTokenizer.from_pretrained(HG_MODEL, revision=MODEL_REVISION,
                                          local_files_only=True, truncation_side="left")
session = ort.InferenceSession(onnx_path, providers=["CoreMLExecutionProvider", "CPUExecutionProvider"])

2026-08-01 15:55:33.758 Python[8094:91518] 2026-08-01 15:55:34.062814 [W:onnxruntime:, coreml_execution_provider.cc:137 GetCapability] CoreMLExecutionProvider::GetCapability, number of partitions supported by CoreML: 195 number of nodes in the graph: 3445 number of nodes supported by CoreML: 1695


In [44]:



base_1 = AutoModelForCausalLM.from_pretrained("livekit/turn-detector",
                                               revision="v0.4.1-intl", dtype=torch.float32)
model_v1 = PeftModel.from_pretrained(base_1, "./uz_eou_lora/v1_R/checkpoint-2190")  
model_v1.eval()
base_2 = AutoModelForCausalLM.from_pretrained("livekit/turn-detector",
                                            revision="v0.4.1-intl", dtype=torch.float32)
model_v2 = PeftModel.from_pretrained(base_2, "./uz_eou_lora/v2/checkpoint-1584")
model_v2.eval()

base_3 = AutoModelForCausalLM.from_pretrained("livekit/turn-detector",
                                            revision="v0.4.1-intl", dtype=torch.float32)
model_v3 = PeftModel.from_pretrained(base_3, "./uz_eou_lora/v5_tug'ulgan kun/checkpoint-948")
model_v3.eval()

base_4 = AutoModelForCausalLM.from_pretrained("livekit/turn-detector",
                                            revision="v0.4.1-intl", dtype=torch.float32)
model_v4 = PeftModel.from_pretrained(base_4, "./uz_eou_lora/v6_numbers/checkpoint-2076")
model_v4.eval()

model_full = AutoModelForCausalLM.from_pretrained("./uz_eou_full/checkpoint-1182", dtype=torch.float32)
model_full.eval()


IM_END_ID = tok.convert_tokens_to_ids("<|im_end|>")
def build_text(messages):
    convo = tok.apply_chat_template(messages, add_generation_prompt=False,
                                    add_special_tokens=False, tokenize=False)
    return convo[:convo.rfind("<|im_end|>")]
@torch.no_grad()
def predict_v1(messages):
    enc = tok(build_text(messages), add_special_tokens=False, return_tensors="pt",
              truncation=True, max_length=128)
    enc = {k: v.to(model_v1.device) for k, v in enc.items()}
    lp = torch.log_softmax(model_v1(**enc).logits[0, -1, :].float(), dim=-1)
    return lp[IM_END_ID].exp().item()

def predict_base(messages):   # fine-tune qilinmagan ONNX
    enc = tok(build_text(messages), add_special_tokens=False, return_tensors="np",
              truncation=True, max_length=128)
    out = session.run(None, {"input_ids": enc["input_ids"].astype("int64")})
    return float(out[0].flatten()[-1])

def predict_v2(messages):
    enc = tok(build_text(messages), add_special_tokens=False, return_tensors="pt",
              truncation=True, max_length=128)
    enc = {k: v.to(model_v2.device) for k, v in enc.items()}
    lp = torch.log_softmax(model_v2(**enc).logits[0, -1, :].float(), dim=-1)
    return lp[IM_END_ID].exp().item()
def predict_v3(messages):
    enc = tok(build_text(messages), add_special_tokens=False, return_tensors="pt",
              truncation=True, max_length=128)
    enc = {k: v.to(model_v3.device) for k, v in enc.items()}
    lp = torch.log_softmax(model_v3(**enc).logits[0, -1, :].float(), dim=-1)
    return lp[IM_END_ID].exp().item()
def predict_v4(messages):
    enc = tok(build_text(messages), add_special_tokens=False, return_tensors="pt",
              truncation=True, max_length=128)
    enc = {k: v.to(model_v4.device) for k, v in enc.items()}
    lp = torch.log_softmax(model_v4(**enc).logits[0, -1, :].float(), dim=-1)
    return lp[IM_END_ID].exp().item()
def predict_pt(messages):
    enc = tok(build_text(messages), add_special_tokens=False, return_tensors="pt",
              truncation=True, max_length=128)
    enc = {k: v.to(model_full.device) for k, v in enc.items()}
    lp = torch.log_softmax(model_full(**enc).logits[0, -1, :].float(), dim=-1)
    return lp[IM_END_ID].exp().item()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [45]:


def auc(labels, scores):
    """Mann-Whitney U orqali AUC — sklearn kerak emas."""
    order = np.argsort(scores)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(scores) + 1)
    for v in np.unique(scores):
        m = scores == v
        if m.sum() > 1:
            ranks[m] = ranks[m].mean()
    n_pos, n_neg = (labels == 1).sum(), (labels == 0).sum()
    return (ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

In [64]:
predict_v1([{"role":"assistant","content":"rahmat. endi telefon raqamlarini ayta olasizmi?"}, {"role":"user","content":"56 834 4 4 2"}])
# predict_v3([{"role":"assistant","content":"rahmat. endi passportingizning raqamlarini ayta olasizmi?"}, {"role":"user","content":"Yetti yuz o'n. Yigirma ikki yetmish uch"}])
# predict_v3([{"role":"assistant","content":"rahmat, aziza nazarova. endi tuğilgan sanangizni kun, oy va yil kõrinişida ayta olasizmi?"}, {"role":"user","content":"Yetmish to'qqizinchi yil"}])

0.9128313660621643

In [47]:
predict_v1([{"role":"assistant","content":"rahmat. endi passport raqamlarini ayta olasizmi?"}, {"role":"user","content":"passport seriyam ae 2 0 5 5 77"}])


0.023113206028938293

In [48]:
rows = [json.loads(l) for l in open("./uz_dataset/uz_eou_smoke_test_num.jsonl", encoding="utf-8")]
labels = np.array([r["label"] for r in rows])

p_base = np.array([predict_base(r["messages"]) for r in rows])
p_v1   = np.array([predict_v1(r["messages"])   for r in rows])
p_v2   = np.array([predict_v2(r["messages"])   for r in rows])
p_v3   = np.array([predict_v3(r["messages"])   for r in rows])
p_v4   = np.array([predict_v4(r["messages"])   for r in rows])
p_pt   = np.array([predict_pt(r["messages"])   for r in rows])

def best_bal(probs):
    best = max(((th,
                 ((probs >= th) & (labels == 1)).sum() / max((labels == 1).sum(), 1),
                 ((probs <  th) & (labels == 0)).sum() / max((labels == 0).sum(), 1))
                for th in np.logspace(-5, -0.3, 60)),
               key=lambda x: (x[1] + x[2]) / 2)
    return best

print(f"{'model':6} {'AUC':>7} {'thr':>8} {'TPR':>6} {'TNR':>6} {'bal':>6}")
for name, p in [("base", p_base), ("v1", p_v1), ("v2", p_v2), ("v3", p_v3), ("v4", p_v4), ("pt", p_pt)]:
    th, tpr, tnr = best_bal(p)
    print(f"{name:6} {auc(labels, p):7.4f} {th:8.5f} {tpr:6.3f} {tnr:6.3f} {(tpr+tnr)/2:6.3f} ") 

Context leak detected, CoreAnalytics returned false
Context leak detected, CoreAnalytics returned false
Context leak detected, CoreAnalytics returned false
Context leak detected, CoreAnalytics returned false
Context leak detected, CoreAnalytics returned false
Context leak detected, CoreAnalytics returned false


model      AUC      thr    TPR    TNR    bal
base    0.5091  0.03844  0.778  0.364  0.571 
v1      0.8613  0.34728  0.833  0.800  0.817 
v2      0.8121  0.03200  0.926  0.618  0.772 
v3      0.8646  0.04617  0.926  0.691  0.808 
v4      0.8005  0.02217  0.944  0.618  0.781 
pt      0.8825  0.13879  0.963  0.636  0.800 


In [49]:
print(f"{'lbl':>3} {'base':>8} {'v1':>8} {'v2':>8} {'v3':>8} {'pt':>8}  matn")
for r, b, a, c, d, e in zip(rows, p_base, p_v1, p_v2, p_v3, p_pt):
    print(f"{r['label']:>3} {b:8.5f} {a:8.5f} {c:8.5f} {d:8.5f} {e:8.5f}  {r['messages'][-1]['content'][:45]}")

lbl     base       v1       v2       v3       pt  matn
  0  0.03828  0.14123  0.00000  0.00000  0.00000  men bugun maktabga
  0  0.01169  0.03249  0.00000  0.00001  0.00001  yo'q bilmadim lekin
  0  0.22256  0.34543  0.00000  0.00010  0.00101  gap shundaki
  0  0.25511  0.26370  0.00066  0.00166  0.00063  aytmoqchi edimki
  0  0.03775  0.10982  0.00000  0.00000  0.00000  reysimni bekor qilishdi va
  0  0.00113  0.00007  0.00000  0.00000  0.00000  telefon raqamim 9 1 6 9 5
  1  0.00911  0.63141  0.36148  0.00397  0.98635  telefon raqamim 9 1 6 9 5 3 4 5 6
  1  0.43391  0.65944  0.99989  0.99955  0.99988  soat necha bo'ldi
  1  0.11466  0.75916  0.99999  0.99992  0.99995  menga 1 daqiqa o'ylab ko'ray
  1  0.09470  0.35652  0.31519  0.70591  0.23628  hmm menimcha keladi
  0  0.10343  0.02940  0.00000  0.00000  0.00000  men ertaga soat o'nda chilonzorga
  1  0.22045  0.00554  0.00000  0.04636  1.00000  chilonzorga
  0  0.00110  0.02468  0.00000  0.00001  0.00000  meni vokzaldan olib
  0  0

In [50]:
def compare(text_or_msgs, expected=None):
    msgs = ([{"role": "user", "content": text_or_msgs}]
            if isinstance(text_or_msgs, str) else text_or_msgs)
    old, new = predict_pt(msgs), predict_v4(msgs)
    last = msgs[-1]["content"]
    line = f"{old:8.5f} -> {new:8.5f}   {last}"
    if expected is not None:
        ok_old = "✓" if (old >= 0.41719) == (expected == 1) else "✗"
        ok_new = "✓" if (new >= 0.005)  == (expected == 1) else "✗"
        line = f"{ok_old}{ok_new} {line}"
    print(line)

In [51]:
print("O'ZBEKCHA (1=tugagan, 0=tugallanmagan)")
compare("soat nechchi bo'ldi", 1)
compare("menga bir daqiqa o'ylab ko'ray", 1)
compare("gap shundaki", 0)
compare("men uchishimni bekor qilishdi va", 0)
compare("aytmoqchi edimki", 0)
compare("kartam yo'qolib", 0)
compare("internetim juda", 0)
compare("rahmat hammasi joyida sog' bo'l", 1)
compare([{"role":"assistant","content":"qayerga borasiz"},
         {"role":"user","content":"chilonzorga"}], 1)
compare([{"role":"assistant","content":"rahmat. endi passportingizning raqamlarini ayta olasizmi?"},
         {"role":"user","content":"Yetti yuz o'n. Yigirma ikki. Yetmish uch."},], 1)
compare([{"role":"assistant","content":"rahmat. endi passportingizning ayta olasizmi?"},
         {"role":"user","content":"123"},], 0)
compare([{"role":"assistant","content":"rahmat. endi passportingizning raqamlarini ayta olasizmi?"},
         {"role":"user","content":"A D 1313323"},], 1)
compare([{"role":"assistant","content":"rahmat. endi passportingizni ayta olasizmi?"},
         {"role":"user","content":"A D Yetti yuz o'n. Yigirma ikki. Yetmish uch."},], 1)


O'ZBEKCHA (1=tugagan, 0=tugallanmagan)
✓✓  0.99986 ->  1.00000   soat nechchi bo'ldi
✓✓  0.99982 ->  0.99999   menga bir daqiqa o'ylab ko'ray
✓✓  0.00101 ->  0.00135   gap shundaki
✓✓  0.00000 ->  0.00000   men uchishimni bekor qilishdi va
✓✗  0.00063 ->  0.99579   aytmoqchi edimki
✓✓  0.00024 ->  0.00002   kartam yo'qolib
✓✓  0.00000 ->  0.00000   internetim juda
✓✓  0.53135 ->  0.88434   rahmat hammasi joyida sog' bo'l
✓✓  1.00000 ->  1.00000   chilonzorga
✓✓  0.99818 ->  1.00000   Yetti yuz o'n. Yigirma ikki. Yetmish uch.
✓✓  0.00001 ->  0.00001   123
✓✓  0.99990 ->  1.00000   A D 1313323
✓✓  0.99950 ->  1.00000   A D Yetti yuz o'n. Yigirma ikki. Yetmish uch.


In [52]:
print("INGLIZCHA (unutilmadimi?)")
compare("what time is it", 1)
compare("so basically my flight got cancelled and", 0)
compare("the thing is", 0)
compare("i need to think about that for a moment", 1)

INGLIZCHA (unutilmadimi?)
✗✓  0.14918 ->  0.08220   what time is it
✓✓  0.00077 ->  0.00012   so basically my flight got cancelled and
✓✓  0.00158 ->  0.00037   the thing is
✓✓  0.65559 ->  0.65297   i need to think about that for a moment


In [53]:
import numpy as np
from sklearn.metrics import roc_auc_score


rows = [json.loads(l) for l in open("./uz_dataset/uz_eou_val.jsonl", encoding="utf-8")]
probs = np.array([predict_pt(r["messages"]) for r in rows])
labels = np.array([r["label"] for r in rows])
print("val AUC:", roc_auc_score(labels, probs))

# eng yaxshi threshold (bank uchun TNR muhimroq — chala gapni kesib yubormaslik)
for th in np.logspace(-4, -0.3, 30):
    tpr = ((probs >= th) & (labels == 1)).sum() / max((labels == 1).sum(), 1)
    tnr = ((probs <  th) & (labels == 0)).sum() / max((labels == 0).sum(), 1)
    print(f"{th:.5f}  tpr={tpr:.3f}  tnr={tnr:.3f}  bal={(tpr+tnr)/2:.3f}")


val AUC: 0.9787477169818499
0.00010  tpr=0.994  tnr=0.723  bal=0.858
0.00013  tpr=0.994  tnr=0.730  bal=0.862
0.00018  tpr=0.994  tnr=0.736  bal=0.865
0.00024  tpr=0.994  tnr=0.747  bal=0.870
0.00032  tpr=0.994  tnr=0.756  bal=0.875
0.00043  tpr=0.994  tnr=0.762  bal=0.878
0.00058  tpr=0.994  tnr=0.771  bal=0.883
0.00078  tpr=0.994  tnr=0.777  bal=0.886
0.00105  tpr=0.994  tnr=0.785  bal=0.889
0.00141  tpr=0.994  tnr=0.793  bal=0.893
0.00189  tpr=0.994  tnr=0.794  bal=0.894
0.00253  tpr=0.994  tnr=0.802  bal=0.898
0.00340  tpr=0.994  tnr=0.811  bal=0.902
0.00456  tpr=0.994  tnr=0.816  bal=0.905
0.00611  tpr=0.992  tnr=0.825  bal=0.908
0.00820  tpr=0.992  tnr=0.829  bal=0.911
0.01100  tpr=0.992  tnr=0.835  bal=0.914
0.01476  tpr=0.992  tnr=0.841  bal=0.917
0.01979  tpr=0.991  tnr=0.846  bal=0.918
0.02655  tpr=0.991  tnr=0.852  bal=0.921
0.03562  tpr=0.991  tnr=0.860  bal=0.925
0.04779  tpr=0.986  tnr=0.864  bal=0.925
0.06411  tpr=0.984  tnr=0.872  bal=0.928
0.08600  tpr=0.984  tnr=0.873

In [54]:
# -*- coding: utf-8 -*-
"""
O'zbekcha raqam diktovkasi uchun deterministik "slot-filling" darvoza.

Muammo: EOU (turn-detector) modeli GRAMMATIK tugaganlikni baholaydi, raqam
diktovkasi esa SANASH masalasi ("yetti yuz o'n yigirma ikki" = 5 raqam, passport
7 raqam bo'lishi kerak -> hali tugamagan). Model buni ishonchli qila olmaydi.

Yechim: call-center oqimida qaysi maydon yig'ilayotgani ma'lum (assistant savolini
o'zing boshqarasan). Foydalanuvchi aytgan raqam so'zlarini raqamga aylantirib,
sonini deterministik tekshirasan. Tugaganlikni model emas, shu darvoza hal qiladi.

Asosiy funksiyalar:
    uz_words_to_digits("yetti yuz o'n yigirma ikki")  -> "71022"
    should_endpoint(user_text, collecting="passport_num", eou_prob=...) -> bool

STT ning uch formatini ham qo'llaydi:
    bo'lak-so'z:    "yetti yuz o'n yigirma ikki yetmish uch" -> "7102273"
    raqamma-raqam:  "yetti bir nol ikki ikki yetti uch"      -> "7102273"
    yaxlit raqam:   "7102273"                                 -> "7102273"
    aralash/prefiks:"raqamim 91 695 34 56"                    -> "916953456"
    harflar (passport seriyasi) e'tiborsiz qoldiriladi:
                    "ab 1234567" / "a be bir ikki uch ..."    -> "1234567"

LiveKit oqimiga ulash (soddalashtirilgan):

    collecting = None       # hozir qaysi raqam maydoni yig'ilyapti (None = erkin nutq)

    def on_agent_turn(agent_text):          # agent savol berganda
        nonlocal collecting
        collecting = detect_field(agent_text)   # yoki o'z oqim holatingdan ber

    def on_user_eou(user_text, eou_prob):   # STT to'liq transkript + EOU ehtimoli
        if should_endpoint(user_text, collecting, eou_prob):
            collecting = None               # maydon to'ldi -> holatni tozala
            return respond()
        return keep_listening()             # sukut taymerini uzaytirib kutamiz

Muhim maslahatlar:
  - Raqam maydonlarida sukut taymerini uzaytir (masalan 1.5-2s) — darvoza "kutamiz"
    deganda ham odam raqamni tugatib ulgursin.
  - Kritik maydonlar (passport/karta/jshshir) uchun read-back tasdiq qo'sh:
    "raqamingiz ... to'g'rimi?" — noto'g'ri uzunlik shu yerda ushlanadi.
  - So'z bilan aytilgan raqamlarni ishonchli sanash ID uslublarida ishlaydi
    (raqamma-raqam, 2-xonali juft, yaxlit). Noaniq "yumaloq" o'qishlar
    ("uch yuz sakson olti") tabiatan noaniq — bunda read-back kafolat beradi.
"""

import re
import unicodedata

# ---------------------------------------------------------------------------
# Normalizatsiya (dataset generatoridagi bilan bir xil) — apostrof variantlari
# ASCII ' ga keltiriladi, kichik harf, punktuatsiya olib tashlanadi.
# ---------------------------------------------------------------------------
_APOS = "ʻʼ‘’`´"


def _normalize(text: str) -> str:
    if not text:
        return ""
    for ch in _APOS:
        text = text.replace(ch, "'")
    text = unicodedata.normalize("NFKC", text.lower())
    text = "".join(ch for ch in text
                   if not (unicodedata.category(ch).startswith("P") and ch not in ["'", "-"]))
    return re.sub(r"\s+", " ", text).strip()


# ---------------------------------------------------------------------------
# Raqam-so'z lug'ati. Apostrofsiz variantlar ham (STT apostrofni tushirishi mumkin).
# ---------------------------------------------------------------------------
# Nol (0) ning yozilish/STT variantlari — alohida qaymiz (leading zero uchun)
ZERO_WORDS = {"nol", "no'l", "nul", "noil", "nöl"}
UNITS = {
    "nol": 0, "no'l": 0, "nul": 0, "bir": 1, "ikki": 2, "uch": 3,
    "to'rt": 4, "tort": 4, "besh": 5, "olti": 6, "yetti": 7,
    "sakkiz": 8, "to'qqiz": 9, "toqqiz": 9,
}
TENS = {
    "o'n": 10, "on": 10, "yigirma": 20, "o'ttiz": 30, "ottiz": 30, "qirq": 40,
    "ellik": 50, "oltmish": 60, "yetmish": 70, "sakson": 80,
    "to'qson": 90, "toqson": 90,
}
HUND = {"yuz": 100}
# Diqqat: bu darvoza ID RAQAM ketma-ketligi uchun (passport/telefon/jshshir/karta).
# "ming" (1000) ID o'qishda uchramaydi — u faqat yil/summa uchun; shuning uchun
# lug'atga kiritilmagan (uchrasa chegara so'zi sifatida qaraladi). Yil tugaganligi
# raqam soni bilan emas, kontekst bilan (model) hal qilinadi.

# "lik" qo'shimchasi (operator kodi belgisi): user "to'qson birlik" (91-lik),
# "to'qsonlik" (90-lik) deydi — shu kompaniyaga tegishli degani. Bu raqam GURUHINI
# yakunlaydi: "to'qsonlik olti yuz..." -> 90 | 6.. (aks holda "to'qson olti"=96
# bo'lib qo'shilib ketardi). Qo'shimchani ajratamiz va guruhni majburan yopamiz.
_LIK_SUFFIXES = ("talik", "taligi", "ligi", "lig'i", "lig", "lik")


def _strip_lik(tok: str):
    """Raqam so'zidagi 'lik' qo'shimchasini ajratadi.
    Qaytaradi: (o'zak, True) agar qo'shimcha bo'lsa, aks holda (tok, False).
    'ellik' (50) kabi to'g'ridan-to'g'ri raqamlarga tegmaydi."""
    if tok in UNITS or tok in TENS or tok in HUND:
        return tok, False          # 'ellik', 'birlik'? -> avval to'g'ridan tekshir
    for suf in _LIK_SUFFIXES:
        stem = tok[:-len(suf)]
        if tok.endswith(suf) and (stem in UNITS or stem in TENS or stem in HUND):
            return stem, True
    return tok, False


def uz_words_to_digits(text: str) -> str:
    """O'zbekcha aytilgan raqamni raqam SATRiga aylantiradi (leading zero saqlanadi).
    Qaytadagi satr uzunligi = aytilgan raqamlar soni. Kirish — STT so'z chiqishi
    ("to'qson birlik olti yuz to'qson besh ..."), literal raqam kutilmaydi (lekin
    kelib qolsa ham qo'llanadi)."""
    tokens = _normalize(text).split()
    out = []              # yig'ilgan raqam bo'laklari
    cur = 0               # joriy guruh qiymati
    seen_tens = False
    seen_units = False
    active = False

    def flush():
        nonlocal cur, seen_tens, seen_units, active
        if active:
            out.append(str(cur))
        cur, seen_tens, seen_units, active = 0, False, False, False

    for raw in tokens:
        if raw.isdigit():                 # STT'da kutilmaydi; xavfsizlik uchun qoldirilgan
            flush()
            out.append(raw)
            continue
        tok, force_flush = _strip_lik(raw)
        if tok in ZERO_WORDS:             # 0 doim yangi guruh boshlaydi (leading zero)
            flush()
            out.append("0")
        elif tok in UNITS:
            if seen_units:                # ketma-ket birlik -> yangi guruh (raqamma-raqam)
                flush()
            cur += UNITS[tok]
            seen_units, active = True, True
        elif tok in TENS:
            if seen_tens or seen_units:   # o'nlik allaqachon bor edi -> yangi guruh
                flush()
            cur += TENS[tok]
            seen_tens, active = True, True
        elif tok in HUND:                 # "olti yuz" = 6*100
            if not active:
                cur = 1
            cur *= 100
            seen_tens = seen_units = False
            active = True
        else:                             # harf / to'ldiruvchi so'z -> guruh chegarasi
            flush()
            continue
        if force_flush:                   # 'lik' -> operator kodi guruhi yopildi
            flush()
    flush()
    return "".join(out)


In [55]:
import sys
from pathlib import Path
# darvoza kodini import qil (uz_words_to_digits, EXPECTED shu yerda)
sys.path.insert(0, "./")   # gate fayli qayerda bo'lsa shu yo'l
from number import uz_words_to_digits, _normalize, UNITS, TENS, HUND, ZERO_WORDS  # fayl nomini moslashtir
EXPECTED = {
    "passport_num": 7,    # passport/ID raqami (seriya harflarisiz)
    "phone": 9,           # operator kodi (2) + 7  — 998 prefiksisiz
    "jshshir": 14,        # PINFL
    "card": 16,           # Uzcard/Humo
    "card_last4": 4,      # oxirgi 4 raqam
}
def _count_number_words(text):
    """Matndagi raqam-so'zlar ulushini qaytaradi (0.0-1.0)."""
    words = _normalize(text).split()
    if not words:
        return 0.0
    num_words = sum(1 for w in words
                    if w in UNITS or w in TENS or w in HUND or w in ZERO_WORDS)
    return num_words / len(words)

def predict_pt2(messages, collecting=None, eou_threshold=0.005):
    """
    Model + raqam-darvoza birlashtirilgan bashorat.

    collecting: yig'ilayotgan raqam maydoni (EXPECTED kaliti) yoki None.
      - Raqam maydoni bo'lsa: raqam soni yetganini darvoza tekshiradi.
      - Aks holda: model qaror qiladi.

    Qaytaradi: (eou_prob, decision, info) — ehtimol, qaror (bool), tafsilot.
    """
    user_text = messages[-1]["content"]

    # modelning xom bashorati (har doim hisoblaymiz — ma'lumot uchun)
    eou_prob = predict_pt(messages)

    # raqamni sana
    digits = uz_words_to_digits(user_text)
    n_digits = len(digits)
    num_ratio = _count_number_words(user_text)

    info = {"digits": digits, "n_digits": n_digits,
            "num_ratio": round(num_ratio, 2), "eou_prob": round(eou_prob, 5)}

    # QAROR MANTIG'I
    if collecting in EXPECTED:
        need = EXPECTED[collecting]
        if n_digits < need:
            return eou_prob, False, {**info, "reason": f"chala raqam {n_digits}/{need}"}
        if num_ratio > 0.7:   # asosan raqam -> darvoza qaror qiladi
            return eou_prob, True, {**info, "reason": f"raqam to'ldi {n_digits}/{need}"}
        # raqam + matn aralash -> modelga ishon
        return eou_prob, eou_prob >= eou_threshold, {**info, "reason": "aralash -> model"}

    # raqam maydoni emas -> sof model
    return eou_prob

In [56]:
print(uz_words_to_digits("oltmish"))

60


In [57]:
predict_pt([{"role": "assistant", "content": "kartangizning oxirgi to'rt raqamini ayting"}, {"role": "user", "content": "ozgina kuting"}, {"role": "assistant", "content": "xo'p kutaman"}, {"role": "user", "content": "6"}])

0.99997878074646

In [58]:
# raqam maydoni (passport = 7 raqam kerak)
prob, decision, info = predict_pt2(
    [{"role": "user", "content": "yetti yuz o'n yigirma ikki"}],
    collecting="passport_num")
print(f"qaror={decision}  {info}")
# -> raqam: "71022", n=5 < 7 -> False (kutamiz), reason "chala raqam 5/7"

prob, decision, info = predict_pt2(
    [{"role": "user", "content": "yetti yuz o'n yigirma ikki yetmish uch"}],
    collecting="passport_num")
print(f"qaror={decision}  {info}")
# -> raqam: "7102273", n=7 >= 7 -> True (tugadi), reason "raqam to'ldi 7/7"

# erkin nutq (raqam maydoni emas)
prob = predict_pt2(
    [{"role": "user", "content": "gap shundaki"}])
print(f"qaror={decision}  {info}")
# -> model qaror qiladi

qaror=False  {'digits': '71022', 'n_digits': 5, 'num_ratio': 1.0, 'eou_prob': 0.0177, 'reason': 'chala raqam 5/7'}
qaror=True  {'digits': '7102273', 'n_digits': 7, 'num_ratio': 1.0, 'eou_prob': 0.94947, 'reason': "raqam to'ldi 7/7"}
qaror=True  {'digits': '7102273', 'n_digits': 7, 'num_ratio': 1.0, 'eou_prob': 0.94947, 'reason': "raqam to'ldi 7/7"}


In [59]:
def compare2(text_or_msgs, expected=None):
    msgs = ([{"role": "user", "content": text_or_msgs}]
            if isinstance(text_or_msgs, str) else text_or_msgs)
    old, new = predict_pt2(msgs), predict_v4(msgs)
    last = msgs[-1]["content"]
    line = f"{old:8.5f} -> {new:8.5f}   {last}"
    if expected is not None:
        ok_old = "✓" if (old >= 0.41719) == (expected == 1) else "✗"
        ok_new = "✓" if (new >= 0.005)  == (expected == 1) else "✗"
        line = f"{ok_old}{ok_new} {line}"
    print(line)

In [60]:
print("O'ZBEKCHA (1=tugagan, 0=tugallanmagan)")
compare2("soat nechchi bo'ldi", 1)
compare2("menga bir daqiqa o'ylab ko'ray", 1)
compare2("gap shundaki", 0)
compare2("men uchishimni bekor qilishdi va", 0)
compare2("aytmoqchi edimki", 0)
compare2("kartam yo'qolib", 0)
compare2("internetim juda", 0)
compare2("rahmat hammasi joyida sog' bo'l", 1)
compare2([{"role":"assistant","content":"qayerga borasiz"},
         {"role":"user","content":"chilonzorga"}], 1)
compare2([{"role":"assistant","content":"rahmat. endi passportingizning raqamlarini ayta olasizmi?"},
         {"role":"user","content":"Yetti yuz o'n. Yigirma ikki. Yetmish uch."},], 1)
compare2([{"role":"assistant","content":"rahmat. endi passportingizning ayta olasizmi?"},
         {"role":"user","content":"Yetti yuz o'n. Yigirma ikki"},], 0)
compare2([{"role":"assistant","content":"rahmat. endi passportingizning raqamlarini ayta olasizmi?"},
         {"role":"user","content":"A D Yetti yuz o'n. Yigirma ikki. Yetmish uch."},], 1)
compare2([{"role":"assistant","content":"rahmat. endi passportingizni ayta olasizmi?"},
         {"role":"user","content":"A D Yetti yuz o'n. Yigirma ikki. Yetmish uch."},], 1)


O'ZBEKCHA (1=tugagan, 0=tugallanmagan)
✓✓  0.99986 ->  1.00000   soat nechchi bo'ldi
✓✓  0.99982 ->  0.99999   menga bir daqiqa o'ylab ko'ray
✓✓  0.00101 ->  0.00135   gap shundaki
✓✓  0.00000 ->  0.00000   men uchishimni bekor qilishdi va
✓✗  0.00063 ->  0.99579   aytmoqchi edimki
✓✓  0.00024 ->  0.00002   kartam yo'qolib
✓✓  0.00000 ->  0.00000   internetim juda
✓✓  0.53135 ->  0.88434   rahmat hammasi joyida sog' bo'l
✓✓  1.00000 ->  1.00000   chilonzorga
✓✓  0.99818 ->  1.00000   Yetti yuz o'n. Yigirma ikki. Yetmish uch.
✓✓  0.10839 ->  0.00303   Yetti yuz o'n. Yigirma ikki
✓✓  0.99957 ->  1.00000   A D Yetti yuz o'n. Yigirma ikki. Yetmish uch.
✓✓  0.99950 ->  1.00000   A D Yetti yuz o'n. Yigirma ikki. Yetmish uch.
